# ProWrite AI — Correction et reformulation avec un modèle Seq2Seq

**Projet NLP | Encoder–Decoder construit from scratch en PyTorch**  
**Tâche réellement évaluée :** transformer une phrase anglaise en une version corrigée, paraphrasée, plus claire ou plus cohérente.

**Question scientifique :** une architecture **BiLSTM + attention additive** améliore-t-elle les résultats d'un **LSTM Encoder–Decoder simple** ?

Ce notebook est conçu comme un livrable académique reproductible : les modèles sont initialisés aléatoirement et entraînés ici. Aucun modèle génératif préentraîné ni API n'est utilisé.

> **Limite honnête.** CoEdIT ne contient pas les paires nécessaires pour transformer systématiquement un brouillon en **e-mail professionnel complet**. Cette fonctionnalité reste une perspective future. Les sorties actuelles sont des transformations de phrases, pas des e-mails fabriqués par un modèle non entraîné à cette tâche.

## Plan
1. Configuration et reproductibilité
2. Corpus, audit et découpage sans fuite de sources identiques
3. Nettoyage, tokenisation et vocabulaire
4. DataLoader et padding
5. Deux modèles Seq2Seq : LSTM puis BiLSTM + Attention
6. Tests du code, teacher forcing et entraînement
7. Décodage greedy et beam search
8. Évaluation quantitative et qualitative
9. Discussion des erreurs, limites et conclusion

## 1. Configuration

Dans Colab, ajoute `coedit_train.csv` et `coedit_validation.csv` à l'onglet Files, ou modifie leurs chemins. Utilise **Runtime → Change runtime type → GPU** si possible. On commence avec un échantillon raisonnable pour une exécution rapide ; augmente `MAX_EXAMPLES` ensuite.

In [ ]:
import os, re, json, math, random, copy, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

TRAIN_CSV = '/content/coedit_train.csv'
VALID_CSV = '/content/coedit_validation.csv'
OUT_DIR = Path('/content/prowrite_results'); OUT_DIR.mkdir(exist_ok=True)

TASKS = ['gec','paraphrase','coherence','clarity']
MAX_EXAMPLES = 16000        # None = toutes les données pertinentes
MAX_SRC_LEN = 55           # inclut SOS/EOS et le tag de tâche
MAX_TGT_LEN = 55
MIN_FREQ = 2
MAX_VOCAB = 16000
BATCH_SIZE = 64
EMB_DIM = 128
HIDDEN_DIM = 256
DROPOUT = 0.15
EPOCHS = 5                # essayer 10-15 pour le rapport final
LR = 0.001
TEACHER_FORCING = 0.6
PATIENCE = 3
assert os.path.exists(TRAIN_CSV) and os.path.exists(VALID_CSV), 'Ajoute les deux fichiers CoEdIT dans Colab.'

## 2. Dataset : extraction de paires

CoEdIT contient `task`, `src` et `tgt`. La colonne `src` commence souvent par une consigne, par exemple « Fix grammatical errors: ». Nous enlevons cette consigne et ajoutons à la place un **tag contrôlé** (`<GEC>`, `<PARAPHRASE>`, etc.).

La normalisation conserve **les majuscules et la ponctuation** : elles font partie de ce que le modèle doit apprendre à générer. Les données originales restent intactes.

In [ ]:
train_raw = pd.read_csv(TRAIN_CSV)
valid_raw = pd.read_csv(VALID_CSV)
print('Fichiers bruts :', train_raw.shape, valid_raw.shape)
print('Tâches :', train_raw.task.value_counts().to_string())

def remove_instruction(src):
    # Les consignes CoEdIT sont suivies de ': '. Garder le contenu après le premier séparateur.
    src = str(src)
    return src.split(': ', 1)[1] if ': ' in src else src

def normalize(s):
    s = unicodedata.normalize('NFKC', str(s))
    s = s.replace('\u200b','').replace('\xa0',' ')
    return re.sub(r'\s+', ' ', s).strip()

def prepare(df):
    z = df.loc[df.task.isin(TASKS), ['task','src','tgt']].dropna().copy()
    z['source_text'] = z.src.map(remove_instruction).map(normalize)
    z['target_text'] = z.tgt.map(normalize)
    z = z[(z.source_text.str.len()>3) & (z.target_text.str.len()>3)]
    z = z[z.source_text != z.target_text]
    z = z.drop_duplicates(['task','source_text','target_text'])
    return z[['task','source_text','target_text']].reset_index(drop=True)

all_pairs = pd.concat([prepare(train_raw),prepare(valid_raw)], ignore_index=True)
all_pairs = all_pairs.drop_duplicates(['task','source_text','target_text'])
print('Paires préparées :',len(all_pairs))
display(all_pairs.sample(min(5,len(all_pairs)),random_state=SEED))

### Découpage train / validation / test

On répartit les données **par phrase source**, pas simplement par ligne : une même phrase source ne doit pas figurer à la fois dans l'entraînement et le test, même si elle apparaît avec deux cibles. On construit les trois partitions **avant de bâtir le vocabulaire**. Les jeux fournis par CoEdIT sont regroupés puis repartitionnés pour garantir cette règle dans notre protocole ; cela doit être indiqué dans le rapport.

In [ ]:
# Si MAX_EXAMPLES est défini, limiter de manière reproductible et stratifiée par tâche.
if MAX_EXAMPLES and len(all_pairs)>MAX_EXAMPLES:
    selected = []
    counts = all_pairs.task.value_counts()
    for task, group in all_pairs.groupby('task'):
        n = max(1, round(MAX_EXAMPLES*len(group)/len(all_pairs)))
        selected.append(group.sample(n=min(n,len(group)), random_state=SEED))
    all_pairs = pd.concat(selected,ignore_index=True)

all_pairs['source_key'] = all_pairs.source_text.str.casefold().str.strip()
keys = all_pairs.source_key.unique().tolist()
rng = np.random.default_rng(SEED); rng.shuffle(keys)
n_train = int(0.8*len(keys)); n_valid = int(0.1*len(keys))
train_keys = set(keys[:n_train]); valid_keys = set(keys[n_train:n_train+n_valid]); test_keys=set(keys[n_train+n_valid:])
train_df = all_pairs[all_pairs.source_key.isin(train_keys)].copy()
val_df = all_pairs[all_pairs.source_key.isin(valid_keys)].copy()
test_df = all_pairs[all_pairs.source_key.isin(test_keys)].copy()
assert not (train_keys&valid_keys or train_keys&test_keys or valid_keys&test_keys)
print('Train/Valid/Test :', len(train_df),len(val_df),len(test_df))
print('Répartition par tâche :',pd.concat({'train':train_df.task.value_counts(),'validation':val_df.task.value_counts(),'test':test_df.task.value_counts()},axis=1).fillna(0).astype(int))

## 3. Tokenisation, longueur et vocabulaire

Le tokenizer simple sépare les mots et les signes de ponctuation. Les tokens spéciaux sont :

- `<PAD>` : complète les séquences courtes ; la loss l'ignore.
- `<UNK>` : mot inconnu (hors vocabulaire).
- `<SOS>` : début de séquence pour le décodeur.
- `<EOS>` : fin de séquence, critère d'arrêt.

Les tags de tâche sont aussi des tokens réservés. **Le vocabulaire est appris uniquement sur train**, sans information du test.

In [ ]:
SPECIAL = ['<PAD>','<UNK>','<SOS>','<EOS>'] + ['<'+t.upper()+'>' for t in TASKS]
PAD,UNK,SOS,EOS = 0,1,2,3
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:[.,]\d+)*|[^\w\s]",re.UNICODE)
def tokenize(s): return TOKEN_RE.findall(normalize(s))

counter = Counter()
for row in train_df.itertuples():
    counter.update(tokenize(row.source_text)); counter.update(tokenize(row.target_text))
vocab = SPECIAL + [t for t,f in counter.most_common() if f>=MIN_FREQ and t not in SPECIAL][:MAX_VOCAB-len(SPECIAL)]
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}
print('Taille du vocabulaire :',len(vocab),'| Tokens spéciaux :',SPECIAL)

def encode_source(task,text):
    ts = ['<SOS>','<'+task.upper()+'>']+tokenize(text)[:MAX_SRC_LEN-3]+['<EOS>']
    return [stoi.get(t,UNK) for t in ts]
def encode_target(text):
    ts = ['<SOS>']+tokenize(text)[:MAX_TGT_LEN-2]+['<EOS>']
    return [stoi.get(t,UNK) for t in ts]
def detokenize(tokens):
    s=' '.join(tokens)
    s=re.sub(r'\s+([.,!?;:%)\]})])',r'\1',s)
    s=re.sub(r'([({\[])\s+',r'\1',s)
    return s.replace(" n't","n't").replace(" 's","'s").replace(" 're","'re")
print('Exemple :',tokenize("Hello, I can't attend tomorrow."))

## 4. Dataset PyTorch et padding par batch

Chaque batch contient des séquences encodées, remplies avec `<PAD>` jusqu'à la plus longue séquence du batch (pas systématiquement jusqu'à la longueur maximale globale). Les longueurs réelles sont conservées pour l'encodeur.

In [ ]:
class TextPairs(Dataset):
    def __init__(self,df): self.rows=list(df[['task','source_text','target_text']].itertuples(index=False,name=None))
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        task,src,tgt=self.rows[i]
        return encode_source(task,src),encode_target(tgt)

def collate(batch):
    sources,targets=zip(*batch)
    src_len=torch.tensor([len(s) for s in sources],dtype=torch.long)
    sx=torch.full((len(batch),max(src_len)),PAD,dtype=torch.long)
    ty=torch.full((len(batch),max(map(len,targets))),PAD,dtype=torch.long)
    for i,(s,t) in enumerate(batch):
        sx[i,:len(s)]=torch.tensor(s); ty[i,:len(t)]=torch.tensor(t)
    return sx,src_len,ty

train_loader=DataLoader(TextPairs(train_df),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate)
val_loader=DataLoader(TextPairs(val_df),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)
test_loader=DataLoader(TextPairs(test_df),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)
sx,sl,ty=next(iter(train_loader))
print('Batch dimensions :',sx.shape,sl.shape,ty.shape)
assert sx.shape[0]==ty.shape[0] and sx.shape[1]<=MAX_SRC_LEN and ty.shape[1]<=MAX_TGT_LEN

## 5. Les architectures de génération

**Modèle A (baseline)** : Embedding → LSTM encodeur unidirectionnel → LSTM décodeur → Linear → Softmax implicite via CrossEntropyLoss.

**Modèle B (amélioration)** : Embedding → **BiLSTM** → **attention additive de Bahdanau** → LSTM décodeur. L'attention pondère les états de l'encodeur à chaque mot généré.

Mathématiquement : $e_{t,i}=v^T\tanh(W_h h_i + W_s s_t)$ ; $\alpha_{t,i}=\operatorname{softmax}_i(e_{t,i})$ ; $c_t=\sum_i\alpha_{t,i}h_i$.

Le décodeur reçoit le token précédent et, pour le modèle B, le vecteur de contexte $c_t$. Un pont linéaire convertit la sortie de l'encodeur (éventuellement bidirectionnelle) en état initial du décodeur.

In [ ]:
class Encoder(nn.Module):
    def __init__(self,vocab_size,emb,hidden,bidirectional=False):
        super().__init__()
        self.bidirectional=bidirectional
        self.embedding=nn.Embedding(vocab_size,emb,padding_idx=PAD)
        self.lstm=nn.LSTM(emb,hidden,batch_first=True,bidirectional=bidirectional)
        enc_dim=hidden*(2 if bidirectional else 1)
        self.h_bridge=nn.Linear(enc_dim,hidden)
        self.c_bridge=nn.Linear(enc_dim,hidden)
    def forward(self,x,lengths):
        embedded=self.embedding(x)
        packed=nn.utils.rnn.pack_padded_sequence(embedded,lengths.cpu(),batch_first=True,enforce_sorted=False)
        out,(h,c)=self.lstm(packed)
        out,_=nn.utils.rnn.pad_packed_sequence(out,batch_first=True,total_length=x.size(1))
        if self.bidirectional:
            h=torch.cat([h[-2],h[-1]],dim=1)
            c=torch.cat([c[-2],c[-1]],dim=1)
        else:
            h,c=h[-1],c[-1]
        return out,(torch.tanh(self.h_bridge(h)).unsqueeze(0),torch.tanh(self.c_bridge(c)).unsqueeze(0))

class AdditiveAttention(nn.Module):
    def __init__(self,enc_dim,hidden):
        super().__init__()
        self.enc=nn.Linear(enc_dim,hidden,bias=False)
        self.dec=nn.Linear(hidden,hidden,bias=False)
        self.score=nn.Linear(hidden,1,bias=False)
    def forward(self,encoder_outputs,hidden,mask):
        energy=self.score(torch.tanh(self.enc(encoder_outputs)+self.dec(hidden[-1]).unsqueeze(1))).squeeze(-1)
        energy=energy.masked_fill(~mask,-1e9)
        weights=torch.softmax(energy,dim=1)
        context=torch.bmm(weights.unsqueeze(1),encoder_outputs).squeeze(1)
        return context,weights

class Decoder(nn.Module):
    def __init__(self,vocab_size,emb,hidden,enc_dim,use_attention):
        super().__init__()
        self.use_attention=use_attention
        self.embedding=nn.Embedding(vocab_size,emb,padding_idx=PAD)
        self.attention=AdditiveAttention(enc_dim,hidden) if use_attention else None
        self.lstm=nn.LSTM(emb+(enc_dim if use_attention else 0),hidden,batch_first=True)
        self.output=nn.Linear(hidden+(enc_dim if use_attention else 0),vocab_size)
    def forward_step(self,token,state,enc_out,mask):
        emb=self.embedding(token).unsqueeze(1)
        if self.use_attention:
            ctx,weights=self.attention(enc_out,state[0],mask)
            lstm_input=torch.cat([emb,ctx.unsqueeze(1)],dim=2)
        else:
            ctx=None; weights=None; lstm_input=emb
        out,state=self.lstm(lstm_input,state)
        rep=torch.cat([out.squeeze(1),ctx],dim=1) if self.use_attention else out.squeeze(1)
        return self.output(rep),state,weights

class Seq2Seq(nn.Module):
    def __init__(self,vocab_size,emb,hidden,bidirectional=False,attention=False):
        super().__init__()
        enc_dim=hidden*(2 if bidirectional else 1)
        self.encoder=Encoder(vocab_size,emb,hidden,bidirectional)
        self.decoder=Decoder(vocab_size,emb,hidden,enc_dim,attention)
    def forward(self,src,lengths,tgt,teacher_forcing=0.5):
        enc_out,state=self.encoder(src,lengths)
        mask=src.ne(PAD)
        token=tgt[:,0]
        outputs=[]
        for step in range(1,tgt.size(1)):
            logits,state,_=self.decoder.forward_step(token,state,enc_out,mask)
            outputs.append(logits)
            token=tgt[:,step] if random.random()<teacher_forcing else logits.argmax(dim=1)
        return torch.stack(outputs,dim=1)

baseline=Seq2Seq(len(vocab),EMB_DIM,HIDDEN_DIM).to(DEVICE)
advanced=Seq2Seq(len(vocab),EMB_DIM,HIDDEN_DIM,bidirectional=True,attention=True).to(DEVICE)
for name,model in [('LSTM',baseline),('BiLSTM + Attention',advanced)]:
    print(name, 'paramètres :', sum(p.numel() for p in model.parameters()))

### Contrôles avant entraînement

Il est important de tester les dimensions, la propagation arrière et la normalisation de l'attention avant de lancer un entraînement coûteux.

In [ ]:
sx,sl,ty=next(iter(train_loader)); sx,sl,ty=sx[:4].to(DEVICE),sl[:4],ty[:4].to(DEVICE)
for m in (baseline,advanced):
    logits=m(sx,sl,ty,teacher_forcing=1.)
    assert logits.shape==(4,ty.size(1)-1,len(vocab))
    loss=nn.CrossEntropyLoss(ignore_index=PAD)(logits.reshape(-1,len(vocab)),ty[:,1:].reshape(-1))
    assert torch.isfinite(loss)
    loss.backward(); m.zero_grad(set_to_none=True)
print('Dimensions + backward : OK')
with torch.no_grad():
    enc,st=advanced.encoder(sx,sl)
    ctx,weights=advanced.decoder.attention(enc,st[0],sx.ne(PAD))
    assert torch.allclose(weights.sum(1),torch.ones(4,device=DEVICE),atol=1e-5)
print('Attention : somme des poids = 1')

## 6. Entraînement (teacher forcing, clipping et early stopping)

Le **teacher forcing** fournit parfois le vrai mot précédent au décodeur pendant l'entraînement. La loss est une **cross-entropy par token**, en ignorant le padding. On limite la norme des gradients pour stabiliser le LSTM et on conserve les meilleurs poids selon la validation.

Pour comparer les deux modèles honnêtement, ils ont les **mêmes données, optimiseur, budget d'époques et procédure de mesure**. Les pertes de validation sont calculées avec teacher forcing = 1 pour être comparables ; la génération réelle sera mesurée séparément.

In [ ]:
criterion=nn.CrossEntropyLoss(ignore_index=PAD)
def epoch_pass(model,loader,optimizer=None,teacher_forcing=1.0):
    training=optimizer is not None
    model.train(training)
    sum_loss=0.; total_tokens=0
    for src,lengths,tgt in loader:
        src=src.to(DEVICE); tgt=tgt.to(DEVICE)
        with torch.set_grad_enabled(training):
            logits=model(src,lengths,tgt,teacher_forcing=teacher_forcing)
            gold=tgt[:,1:]
            loss=criterion(logits.reshape(-1,len(vocab)),gold.reshape(-1))
            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.)
                optimizer.step()
        n=gold.ne(PAD).sum().item()
        sum_loss+=loss.item()*n; total_tokens+=n
    return sum_loss/max(1,total_tokens)

def train_model(model,name):
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    history={'train':[],'validation':[]}
    best=float('inf'); waits=0
    filename=OUT_DIR/(name+'.pt')
    for epoch in range(1,EPOCHS+1):
        begin=time.time()
        train_loss=epoch_pass(model,train_loader,opt,teacher_forcing=TEACHER_FORCING)
        val_loss=epoch_pass(model,val_loader,None,teacher_forcing=1.)
        history['train'].append(train_loss);history['validation'].append(val_loss)
        print(f'{name} | epoch {epoch}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | {time.time()-begin:.0f}s')
        if val_loss<best-1e-4:
            best=val_loss;waits=0
            torch.save(model.state_dict(),filename)
        else:
            waits+=1
            if waits>=PATIENCE:
                print('Early stopping');break
    model.load_state_dict(torch.load(filename,map_location=DEVICE,weights_only=True))
    return history

# Deux vrais entraînements from scratch : selon GPU et nombre d'exemples, cela peut prendre du temps.
hist_base=train_model(baseline,'baseline')
hist_adv=train_model(advanced,'attention')

### Courbes d'apprentissage

Interprétation : si la loss train baisse mais la loss validation remonte, il peut y avoir surapprentissage. L'effet de l'attention doit être démontré par les valeurs calculées, pas présumé.

In [ ]:
plt.figure(figsize=(10,4))
for name,h in [('LSTM',hist_base),('BiLSTM+Attention',hist_adv)]:
    plt.plot(range(1,len(h['train'])+1),h['train'],label=name+' train')
    plt.plot(range(1,len(h['validation'])+1),h['validation'],'--',label=name+' validation')
plt.xlabel('Époque');plt.ylabel('Cross-entropy par token');plt.title('Courbes de loss');plt.legend();plt.grid(alpha=.2)
plt.savefig(OUT_DIR/'learning_curves.png',dpi=150,bbox_inches='tight');plt.show()

## 7. Inférence autorégressive

À l'inférence, la cible correcte n'est pas disponible. Le décodeur part de `<SOS>` et produit un mot à la fois jusqu'à `<EOS>` ou la longueur maximale.

- **Greedy** : choisir le mot le plus probable à chaque étape.
- **Beam search** : conserver plusieurs hypothèses possibles pour chercher une meilleure séquence. C'est une comparaison optionnelle plus avancée.

In [ ]:
@torch.no_grad()
def generate(model,task,text,strategy='greedy',beam_size=3,max_len=55):
    model.eval()
    src_ids=encode_source(task,text)
    src=torch.tensor([src_ids],dtype=torch.long,device=DEVICE)
    lengths=torch.tensor([len(src_ids)])
    enc,state=model.encoder(src,lengths)
    mask=src.ne(PAD)
    if strategy=='greedy':
        tokens=[SOS]; attentions=[]
        for _ in range(max_len-1):
            logits,state,w=model.decoder.forward_step(torch.tensor([tokens[-1]],device=DEVICE),state,enc,mask)
            next_id=int(logits.argmax(1).item())
            if next_id==EOS:break
            if next_id not in (PAD,SOS): tokens.append(next_id)
            else: tokens.append(UNK)
            if w is not None: attentions.append(w.squeeze(0).cpu().numpy())
        out=[itos.get(i,'<UNK>') for i in tokens[1:]]
        return detokenize(out),attentions
    elif strategy=='beam':
        # Log-probabilité normalisée par la longueur pour limiter le biais vers les sorties courtes.
        beams=[([SOS],0.,state,False)]
        for _ in range(max_len-1):
            candidates=[]
            for ids,score,st,ended in beams:
                if ended:
                    candidates.append((ids,score,st,True));continue
                logits,next_state,_=model.decoder.forward_step(torch.tensor([ids[-1]],device=DEVICE),st,enc,mask)
                log_probs=torch.log_softmax(logits,dim=-1).squeeze(0)
                values,indices=log_probs.topk(beam_size)
                for value,index in zip(values.tolist(),indices.tolist()):
                    candidates.append((ids+[index],score+value,next_state,index==EOS))
            beams=sorted(candidates,key=lambda b:b[1]/max(1,len(b[0])-1)**0.7,reverse=True)[:beam_size]
            if all(b[3] for b in beams):break
        best=max(beams,key=lambda b:b[1]/max(1,len(b[0])-1)**0.7)[0][1:]
        out=[itos.get(i,'<UNK>') for i in best if i not in (PAD,SOS,EOS)]
        return detokenize(out),[]
    else:
        raise ValueError('strategy = greedy ou beam')

example='She go to school yesterday and forget her book.'
for model_name,model in [('LSTM',baseline),('Attention',advanced)]:
    for method in ['greedy','beam']:
        print(model_name,method,':',generate(model,'gec',example,method)[0])

## 8. Évaluation quantitative sur le test réservé

On rapporte :

- **Exact Match** (même phrase après normalisation légère) ;
- **Token F1** (chevauchement entre mots générés et référence) ;
- **ROUGE-L F1** (plus longue sous-séquence commune) ;
- **BLEU-4** avec lissage, si NLTK est présent ;
- **Perplexité** $\exp(\text{loss test})$, diagnostique du modèle et non mesure directe de qualité humaine.

Les métriques automatiques ne mesurent pas parfaitement le sens ou le professionnalisme. Une analyse manuelle est nécessaire. Pour maîtriser le temps, `EVAL_LIMIT` fixe une taille reproductible du sous-test généré.

In [ ]:
from collections import Counter as Bag
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    HAS_BLEU=True
except ImportError:
    HAS_BLEU=False

def token_f1(pred,ref):
    p=tokenize(pred);r=tokenize(ref)
    if not p or not r:return 0.
    common=sum((Bag(p)&Bag(r)).values())
    return 2*common/(len(p)+len(r))
def rouge_l(pred,ref):
    p=tokenize(pred);r=tokenize(ref)
    if not p or not r:return 0.
    prev=[0]*(len(r)+1)
    for a in p:
        cur=[0]*(len(r)+1)
        for j,b in enumerate(r,1):
            cur[j]=prev[j-1]+1 if a==b else max(prev[j],cur[j-1])
        prev=cur
    lcs=prev[-1]
    return 2*lcs/(len(p)+len(r))
def bleu4(pred,ref):
    if not HAS_BLEU:return float('nan')
    return sentence_bleu([tokenize(ref)],tokenize(pred),smoothing_function=SmoothingFunction().method1)

EVAL_LIMIT=250  # passer à None pour évaluer tout le test ; génération lente
eval_df=test_df.sample(n=min(EVAL_LIMIT,len(test_df)),random_state=SEED) if EVAL_LIMIT else test_df
def evaluate_generations(model,name,strategy='greedy'):
    rows=[]
    for row in eval_df.itertuples():
        pred,_=generate(model,row.task,row.source_text,strategy=strategy)
        ref=row.target_text
        rows.append({'model':name,'strategy':strategy,'task':row.task,'source':row.source_text,'reference':ref,'prediction':pred,
                     'exact_match':float(normalize(pred).casefold()==normalize(ref).casefold()),
                     'token_f1':token_f1(pred,ref),'rouge_l':rouge_l(pred,ref),'bleu4':bleu4(pred,ref)})
    return pd.DataFrame(rows)

results=pd.concat([
    evaluate_generations(baseline,'LSTM','greedy'),
    evaluate_generations(advanced,'BiLSTM + Attention','greedy')
],ignore_index=True)
test_losses={'LSTM':epoch_pass(baseline,test_loader,None,1.),'BiLSTM + Attention':epoch_pass(advanced,test_loader,None,1.)}
summary=results.groupby('model')[['exact_match','token_f1','rouge_l','bleu4']].mean().round(4)
summary['test_loss']=pd.Series(test_losses)
summary['perplexity']=summary.test_loss.map(lambda x:math.exp(min(x,20)))
display(summary)
display(results.groupby(['model','task'])[['token_f1','rouge_l']].mean().round(3))
results.to_csv(OUT_DIR/'test_predictions.csv',index=False)
summary.to_csv(OUT_DIR/'test_metrics.csv')

### Comparer greedy et beam search

Comparer les méthodes sur **les mêmes exemples**, avec le modèle avancé. Ce test est séparé de la comparaison architecturale.

In [ ]:
beam_subset=eval_df.head(min(40,len(eval_df))).copy()
beam_rows=[]
for row in beam_subset.itertuples():
    for strategy in ['greedy','beam']:
        prediction,_=generate(advanced,row.task,row.source_text,strategy=strategy)
        beam_rows.append({'strategy':strategy,'rouge_l':rouge_l(prediction,row.target_text),'token_f1':token_f1(prediction,row.target_text)})
display(pd.DataFrame(beam_rows).groupby('strategy').mean().round(3))

## 9. Analyse qualitative : réussites et échecs

Examiner plusieurs entrées, références et sorties. Dans le rapport, commenter notamment : erreurs d'accord, mots inconnus `<UNK>`, troncature, omission d'informations, changements de sens et incapacité à généraliser aux e-mails complets.

In [ ]:
view=results[results.model=='BiLSTM + Attention'].copy()
view['difficulty']=view.rouge_l
print('Exemples difficiles :')
display(view.nsmallest(10,'difficulty')[['task','source','reference','prediction','rouge_l']])
print('Exemples les mieux notés :')
display(view.nlargest(10,'difficulty')[['task','source','reference','prediction','rouge_l']])

### Visualiser l'attention sur une phrase

La carte montre sur quels mots d'entrée le décodeur se concentre à chaque mot produit. **Attention n'est pas une preuve causale d'explication** ; c'est un outil de diagnostic.

In [ ]:
example_row=test_df.iloc[0]
prediction,weights=generate(advanced,example_row.task,example_row.source_text)
src_tokens=['<SOS>','<'+example_row.task.upper()+'>']+tokenize(example_row.source_text)[:MAX_SRC_LEN-3]+['<EOS>']
out_tokens=tokenize(prediction)
if weights and out_tokens:
    matrix=np.asarray(weights)
    n=min(len(out_tokens),matrix.shape[0])
    plt.figure(figsize=(min(15,max(7,len(src_tokens)*.3)),min(9,max(3,n*.3))))
    plt.imshow(matrix[:n,:len(src_tokens)],aspect='auto',cmap='Blues')
    plt.xticks(range(len(src_tokens)),src_tokens,rotation=90)
    plt.yticks(range(n),out_tokens[:n]);plt.colorbar(label='Poids attention')
    plt.title('Attention additive — exemple du test');plt.tight_layout()
    plt.savefig(OUT_DIR/'attention_example.png',dpi=150,bbox_inches='tight');plt.show()
else:
    print('Aucun mot généré : attention non affichable.')

## 10. Démonstration interactive (facultative dans Colab)

Utiliser les modèles réellement entraînés. Le menu comporte uniquement les **quatre tâches supervisées** disponibles. La génération d'e-mails professionnels reste à développer avec un corpus parallèle adapté.

In [ ]:
def rewrite(text,task='gec',strategy='greedy'):
    return generate(advanced,task,text,strategy=strategy)[0]

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    field=widgets.Textarea(value='I has a meeting tomorow.',description='Texte',layout=widgets.Layout(width='80%',height='75px'))
    task_widget=widgets.Dropdown(options=TASKS,value='gec',description='Tâche')
    strategy_widget=widgets.Dropdown(options=['greedy','beam'],description='Décodage')
    button=widgets.Button(description='Générer',button_style='primary')
    area=widgets.Output()
    def on_click(_):
        with area:
            clear_output(); print(rewrite(field.value,task_widget.value,strategy_widget.value))
    button.on_click(on_click)
    display(field,task_widget,strategy_widget,button,area)
except ImportError:
    print('ipywidgets indisponible : utilisez rewrite("Votre texte", "gec").')

## 11. Sauvegarde et reproductibilité

On conserve le vocabulaire, les hyperparamètres, les poids du meilleur checkpoint et les prédictions. Les fichiers produits ne constituent des résultats finaux qu'**après exécution** du notebook sur les données et les ressources indiquées.

In [ ]:
config={'seed':SEED,'tasks':TASKS,'max_examples':MAX_EXAMPLES,'max_src_len':MAX_SRC_LEN,'max_tgt_len':MAX_TGT_LEN,
        'vocab_size':len(vocab),'embedding':EMB_DIM,'hidden':HIDDEN_DIM,'batch_size':BATCH_SIZE,'epochs_max':EPOCHS,
        'learning_rate':LR,'teacher_forcing':TEACHER_FORCING,'split':[.8,.1,.1], 'eval_limit':EVAL_LIMIT}
(OUT_DIR/'vocab.json').write_text(json.dumps(vocab,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT_DIR/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print('Fichiers :')
for p in sorted(OUT_DIR.iterdir()): print('-',p.name,round(p.stat().st_size/1024,1),'Ko')

## 12. Discussion scientifique à compléter après exécution

### Questions à répondre dans le rapport
1. La loss baisse-t-elle sur train et validation ? Le meilleur modèle sur validation l'est-il aussi sur test ?
2. Quelle est la différence de ROUGE-L / F1 entre LSTM et BiLSTM + Attention ?
3. L'attention améliore-t-elle les phrases plus longues ? Analyser plusieurs cas, sans inférer un effet non mesuré.
4. Beam search est-il meilleur que greedy sur le même sous-test ?
5. Quels types d'erreurs restent fréquents : omissions, répétitions, troncature, mauvaise ponctuation, `<UNK>` ?
6. La différence de nombre de paramètres explique-t-elle une partie de la différence de performance ?

### Limites
- CoEdIT contient plusieurs styles et genres : le modèle n'est pas spécialisé sur les e-mails.
- Un tokenizer de mots simples produit des `<UNK>` sur les termes rares ; un tokenizer sous-mots pourrait améliorer cela.
- Les données sont en anglais et le modèle from scratch n'a pas de connaissance générale préentraînée.
- Une même phrase peut admettre de nombreuses reformulations correctes : BLEU et Exact Match peuvent être sévères.
- Le découpage empêche la répétition exacte des sources entre partitions ; des gabarits ou formulations très proches peuvent encore exister.
- Le sous-test généré est limité par défaut pour gagner du temps : évaluer tout le test pour des conclusions définitives.

### Conclusion (à rédiger avec les vrais chiffres)
Présenter l'objectif, la taille finale des partitions, les hyperparamètres, les pertes et métriques réellement obtenues, trois succès et trois échecs commentés. Ne jamais affirmer une supériorité d'architecture avant la comparaison calculée.

### Perspective ProWrite AI
Construire des paires *brouillon informel avec fautes → message/e-mail professionnel*, avec contrôle des informations et de la confidentialité, puis ajouter des tags `<PRO_MESSAGE>` et `<PRO_EMAIL>` entraînés explicitement.